# **Predicting Future Alliances: A Guide to Graph Neural Networks for Edge Prediction**

In the previous tutorial, we explored the *structure* of the Cold War alliance network by learning node embeddings for each country and clustering them into geopolitical blocs. Now, we'll turn to a different, predictive question: **Can we predict which countries are likely to form an alliance in the future?**

This is an **edge prediction** (or link prediction) task. We will train a **Graph Neural Network (GNN)** to learn the underlying patterns of treaty formation. The GNN will learn to distinguish between pairs of countries that have an alliance (a "positive edge") and those that do not (a "negative edge"). Once trained, this model can score any pair of nations, giving us a powerful tool to forecast potential geopolitical shifts.

For this task, we will use **PyTorch** and the **PyTorch Geometric (PyG)** library, the premier framework for building and training GNNs with the following steps:

  * **Prepare a graph for edge prediction**, including splitting edges and generating negative samples.
  * Construct a graph data object using **PyTorch Geometric**.
  * Implement a **Graph Convolutional Network (GCN)** model for edge prediction.
  * **Train a GNN** to classify real vs. non-existent alliances.
  * **Evaluate the model's predictive power** using the AUC score.
  * Use the trained model to **predict the most likely new alliances** for a given country.

**Credits**

Data provided by the [**Correlates of War Project**](https://correlatesofwar.org/data-sets/formal-alliances/).

**Compatibility**

| Platform                     | Compatible |  Recommended  | Notes                                                                                                                                   |
| :--------------------------- | :--------: | :-----------: | :-------------------------------------------------------------------------------------------------------------------------------------- |
| **Local (e.g., MacBook/PC)** |   ✅ Yes   |   ✅ Yes      | The COW dataset is small and can be processed efficiently on local machine, making it an ideal setup for learning and practicing the workflow.                                          |
| **Google Colab**             |   ✅ Yes   |   ✅ Yes      | Works well. May require installing libraries: `!pip install "dask[complete]" networkx`                                                  |
| **Midway3 Login Node**       |   ✅ Yes   |    ✅ Yes     | Install Dask and run this on any login nodes. Dask will handle the Slurm job provisioning under the hood.                               |
| **Midway3 Compute Node**     |   ✅ Yes   |   ✅ Yes      | Ideal for this tutorial and for scaling up to larger datasets. Use `LocalCluster()` instead.                                            |

## **What is PyTorch Geometric (PyG)?**

**PyTorch Geometric** is an extension library for PyTorch that makes it easy to work with graph-structured data. It provides a set of powerful tools that handle the complexities of GNNs, including:

  * **`Data` Objects:** A specialized data structure that efficiently stores all the components of a graph (nodes, edges, features) in a single object.
  * **Graph-Specific Layers:** A rich collection of pre-built GNN layers, like `GCNConv`, that implement the core "message passing" logic.
  * **Data Loaders & Transformations:** Utilities designed to handle batching and preprocessing of graph data for training.

By using PyG, we can focus on the high-level architecture of our model rather than the low-level implementation details of graph algorithms.

## **Step 1: Environment set Up and Load Data**

We begin by loading the Correlates of War dataset. Instead of using the Dask/RAPIDS stack, we will use **pandas** and **PyTorch**, as PyG is deeply integrated with the standard PyTorch ecosystem. We will map country names to integer IDs, a necessary step for creating the tensors our model will use.

In [ ]:
# Run this cell to install necessary dependencies
# %pip install torch_geometric==2.6.1 plotly==6.3.0

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Module, Linear
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = "../wk5.1-graph-community-detection/alliance_v4.1"
SEED = 42

In [ ]:
def set_seed(seed=SEED):
    """
    Sets the random seeds for deterministic experiments.

    Args:
        seed (int): The seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.

    # These two settings are often needed for deterministic results with CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call the function at the beginning of your script
set_seed(SEED)

In [ ]:
# --- 1. Load and Filter Data with Pandas ---
file_path = os.path.join(DATA_DIR, "alliance_v4.1_by_dyad.csv")
df = pd.read_csv(file_path)

# --- 2. Data Cleaning and Filtering ---
df = df.dropna(subset=['state_name1', 'state_name2'])
df['dyad_end_year'] = df['dyad_end_year'].fillna(2012)
cold_war_df = df[(df['dyad_st_year'] <= 1991) & (df['dyad_end_year'] >= 1947)].copy()

# Create a new column containing a frozenset of the two country codes.
# A frozenset is used because it's "hashable," which is required by drop_duplicates.
cold_war_df['dyad_set'] = cold_war_df.apply(
    lambda row: frozenset([row['ccode1'], row['ccode2']]),
    axis=1
)

# Keep the first occurrence of each unique frozenset and drop the rest
cold_war_df.drop_duplicates(subset=['dyad_set'], keep='first', inplace=True)

# Remove the temporary helper column
cold_war_df.drop(columns=['dyad_set'], inplace=True)

# Display the result
cold_war_df.head()

,version4id,ccode1,state_name1,ccode2,state_name2,dyad_st_day,dyad_st_month,dyad_st_year,dyad_end_day,dyad_end_month,dyad_end_year,left_censor,right_censor,defense,neutrality,nonaggression,entente,asymmetric,version
0,1,200,United Kingdom,235,Portugal,1,1,1816,NaN,NaN,2012.0,1,1,1,0,1,0.0,0,4.1
279,88,130,Ecuador,145,Bolivia,17,4,1911,NaN,NaN,2012.0,0,1,0,1,1,0.0,0,4.1
356,126,365,Russia,700,Afghanistan,31,8,1926,24.0,12.0,1979.0,0,0,0,1,1,0.0,0,4.1
365,135,640,Turkey,700,Afghanistan,25,5,1928,25.0,5.0,1948.0,0,0,0,0,1,0.0,0,4.1
373,143,200,United Kingdom,645,Iraq,3,10,1932,5.0,4.0,1955.0,0,0,1,0,0,1.0,0,4.1


In [ ]:
# --- 3. Map Nodes to Integers ---
# Get unique node names and create the mapping
all_nodes = pd.concat([cold_war_df['state_name1'], cold_war_df['state_name2']]).unique()
node_map = {name: i for i, name in enumerate(all_nodes)}
reverse_node_map = {i: name for name, i in node_map.items()}
num_nodes = len(all_nodes)

# Apply the mapping to create integer source and target columns
cold_war_df['source'] = cold_war_df['state_name1'].map(node_map)
cold_war_df['target'] = cold_war_df['state_name2'].map(node_map)

print(f"Loaded and processed data for {num_nodes} unique nations.")
cold_war_df[['source', 'target']].head()

Loaded and processed data for 139 unique nations.


,source,target
0,0,72
279,1,8
356,2,88
365,3,88
373,0,6


-----

## **Step 2: Preparing Data for Edge Prediction**

This is the most critical step for our predictive task. We can't train and test our model on the same set of alliances. We need to hide some of the real alliances from the model during training and then test if the model can predict them.

PyG's `RandomLinkSplit` transform is perfect for this by automatically:

1.  **Splits Edges:** It divides the existing alliances (positive edges) into three sets: **train**, **validation**, and **test**.
2.  **Generate Negative Edges:** For each set, it creates an equal number of "negative" edges—pairs of countries that are *not* allied—for the model to learn from.
3.  **Create Subgraphs:** The training data will contain all the nodes but only the training edges, preventing the model from "seeing" the validation and test edges.

<!-- end list -->

In [ ]:
# --- 1. Create the Edge Tensor ---
# PyG expects a tensor of shape [2, num_edges]
edge_index = torch.tensor(cold_war_df[['source', 'target']].values, dtype=torch.long).t().contiguous()

# --- 2. Create the PyG Data Object ---
# We'll use a simple identity matrix for node features, allowing the GNN
# to learn embeddings from scratch based only on graph structure.
node_features = torch.eye(num_nodes)
data = Data(x=node_features, edge_index=edge_index)

# --- 3. Create Training, Validation, and Test Splits ---
transform = RandomLinkSplit(
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0,
)

train_data, val_data, test_data = transform(data)

print("Original Graph:", data)
print("\nTraining Data:", train_data)
print("\nValidation Data:", val_data)

Original Graph: Data(x=[139, 139], edge_index=[2, 1348])

Training Data: Data(x=[139, 139], edge_index=[2, 1322], edge_label=[1322], edge_label_index=[2, 1322])

Validation Data: Data(x=[139, 139], edge_index=[2, 1322], edge_label=[188], edge_label_index=[2, 188])


As you can see from the output, `train_data` contains the edges and edge labels for training, while `val_data` and `test_data` contain the positive and negative edges we will use for evaluation.

-----

## **Step 3: Building a GNN Model for Edge Prediction**

Our model will have two main components:

1.  **Encoder:** This part uses two `GCNConv` layers to process the graph structure. Each node "talks" to its neighbors, aggregating information to produce a rich, 64-dimensional embedding for every country. This is similar to what Node2Vec does, but it's learned end-to-end as part of our prediction task.
2.  **Decoder:** This part takes the embeddings of two nodes (a potential edge) and calculates a score representing the probability of an alliance. We'll use a simple dot product for this: if two countries' embeddings point in a similar direction, their dot product will be high, indicating a likely link.

<!-- end list -->

In [ ]:
class LinkPredictor(Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Encoder layers to learn node embeddings
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def encode(self, x, edge_index):
        """
        Takes node features and the graph structure and returns node embeddings.
        """
        # Apply GCN layers with ReLU activation
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        """
        Takes node embeddings (z) and a set of edges and predicts their existence.
        """
        # Dot product between source and target node embeddings
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def decode_all(self, z):
        """
        Calculates scores for all possible pairs of nodes.
        """
        prob_adj = z @ z.t() # Matrix multiplication for all-pairs dot product
        return prob_adj

# --- Model Initialization ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LinkPredictor(
    in_channels=num_nodes,      # Input feature size is num_nodes (identity matrix)
    hidden_channels=128,
    out_channels=64             # We'll learn 64-dimensional embeddings
).to(device)

# Move all data to the selected device
data = data.to(device)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss() # Perfect for binary classification

Using device: cuda


-----

## **Step 4: The Training and Evaluation Loop**

Now we'll write the code to train our GNN. In each epoch, we will:

1.  Generate node embeddings using the `encode` method on the **training graph**.
2.  Get prediction scores for the training edges (both positive and negative) using the `decode` method.
3.  Calculate the binary cross-entropy loss between our predictions and the true labels.
4.  Update the model's weights using backpropagation.

After each epoch, we'll run an evaluation function on the **validation set** to check how well the model generalizes to unseen data. Our key metric here will be **AUC (Area Under the ROC Curve)**, which measures the model's ability to correctly rank positive edges higher than negative edges. An AUC of 1.0 is a perfect classifier, while 0.5 is no better than random guessing.

In [ ]:
def train(model):
    model.train()
    optimizer.zero_grad()

    # Use the training graph to learn embeddings
    z = model.encode(train_data.x, train_data.edge_index)

    # Get predictions for the training edges
    out = model.decode(z, train_data.edge_label_index)

    # Compare predictions to true labels
    loss = criterion(out, train_data.edge_label)
    loss.backward()
    optimizer.step()
    return loss

@torch.no_grad()
def test(model, data_split):
    model.eval()

    # Get the embeddings from the training graph
    z = model.encode(train_data.x, train_data.edge_index)

    # Get predictions for the edges in the validation or test split
    out = model.decode(z, data_split.edge_label_index)

    # Calculate the AUC score
    return roc_auc_score(data_split.edge_label.cpu().numpy(), out.sigmoid().cpu().numpy())

# --- Training Loop ---
best_val_auc = 0
for epoch in range(1, 21):
    loss = train(model)
    val_auc = test(model, val_data)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch: {epoch}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}')

# --- Final Evaluation on Test Set ---
# Load the best performing model
model.load_state_dict(torch.load('best_model.pt'))
test_auc = test(model, test_data)
print(f'\nFinal Test AUC: {test_auc:.4f}')

Epoch: 1, Loss: 0.6921, Val AUC: 0.8835
Epoch: 2, Loss: 0.6810, Val AUC: 0.8717
Epoch: 3, Loss: 0.6542, Val AUC: 0.8689
Epoch: 4, Loss: 0.6087, Val AUC: 0.8794
Epoch: 5, Loss: 0.5468, Val AUC: 0.8908
Epoch: 6, Loss: 0.4768, Val AUC: 0.8914
Epoch: 7, Loss: 0.4629, Val AUC: 0.8891
Epoch: 8, Loss: 0.4866, Val AUC: 0.8912
Epoch: 9, Loss: 0.4606, Val AUC: 0.8967
Epoch: 10, Loss: 0.4331, Val AUC: 0.9022
Epoch: 11, Loss: 0.4245, Val AUC: 0.9082
Epoch: 12, Loss: 0.4194, Val AUC: 0.9128
Epoch: 13, Loss: 0.4206, Val AUC: 0.9178
Epoch: 14, Loss: 0.4193, Val AUC: 0.9260
Epoch: 15, Loss: 0.4148, Val AUC: 0.9308
Epoch: 16, Loss: 0.4119, Val AUC: 0.9340
Epoch: 17, Loss: 0.4085, Val AUC: 0.9334
Epoch: 18, Loss: 0.4027, Val AUC: 0.9312
Epoch: 19, Loss: 0.3956, Val AUC: 0.9265
Epoch: 20, Loss: 0.3906, Val AUC: 0.9198

Final Test AUC: 0.9440


With a final Test AUC score significantly higher than 0.5, our model has clearly learned meaningful patterns from the alliance network structure\!

-----

## **Step 5: Making Predictions**

Now for the fun part. Let's apply the trained model to find the most likely future allies for a specific country with the following steps:
1.  Generate the final embeddings for all countries using the full graph.
2.  Calculate the prediction scores for **all possible pairs** of nodes.
3.  For Switzerland, find the non-allied countries with the highest scores.

<!-- end list -->

In [ ]:
# --- Generate final embeddings using the full graph ---
model.eval()
with torch.no_grad():
    final_z = model.encode(data.x, data.edge_index)
    final_scores = model.decode_all(final_z).sigmoid()

import numpy as np # Needed for efficient min/max calculation

def predict_new_alliances(country_name, top_n=10, normalize=False):
    """
    Predicts new alliances for a given country, with an option to normalize scores.
    """
    if country_name not in node_map:
        print(f"Country '{country_name}' not found.")
        return

    node_id = node_map[country_name]
    country_scores = final_scores[node_id]

    # Find existing allies to exclude them
    source_edges = cold_war_df[cold_war_df['source'] == node_id]
    target_edges = cold_war_df[cold_war_df['target'] == node_id]
    existing_allies = set(source_edges['target']).union(set(target_edges['source']))

    # Collect all potential new allies and their raw scores
    predictions = []
    for i, score in enumerate(country_scores.cpu().numpy()):
        if i != node_id:
            predictions.append((reverse_node_map[i], score))

    # Sort by score in descending order and return the top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:top_n]

# --- Get predictions for another country ---
japan_preds = predict_new_alliances("Japan", top_n=10, normalize=True)
print("\nTop 10 most likely new alliances for Japan:")
for country, score in japan_preds:
    print(f"- {country}: {score:.4f}")


Top 10 most likely new alliances for Japan:
- Iceland: 0.9611
- Denmark: 0.9597
- Norway: 0.9581
- Turkey: 0.9572
- Greece: 0.9550
- Italy: 0.9543
- German Federal Republic: 0.9517
- Germany: 0.9489
- Portugal: 0.9455
- Spain: 0.9413


In [ ]:
import plotly.express as px
import pandas as pd

def visualize_alliance_scores(predictions, source_country_name):
    """
    Visualizes predicted alliance scores on a world map using Plotly.

    Args:
        predictions (list): A list of tuples, where each tuple contains a
                            country name and a prediction score.
        source_country_name (str): The name of the country for which predictions
                                   were made, used for the map's title.
    """
    # Convert the prediction data into a pandas DataFrame
    pred_df = pd.DataFrame(predictions, columns=['Country', 'Score'])

    # Create an interactive choropleth map
    fig = px.choropleth(
        pred_df,
        locations="Country",
        locationmode="country names",  # Use country names to find locations
        color="Score",                 # The value to represent with color
        hover_name="Country",          # Display country name on hover
        color_continuous_scale=px.colors.sequential.Viridis, # Color scheme
        title=f"Potential Alliances for {source_country_name}"
    )

    # Customize the map's appearance
    fig.update_layout(
        title_x=0.5, # Center the title
        geo=dict(
            showframe=False,
            showcoastlines=False,
            projection_type='equirectangular' # A common map projection
        )
    )

    fig.show()

dprk_preds = predict_new_alliances("North Korea", top_n=100, normalize=True)
visualize_alliance_scores(dprk_preds, "North Korea")

/scratch/local/jobs/35877280/ipykernel_3700182/1883604532.py:18: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [ ]:
# First, generate the predictions
us_preds = predict_new_alliances("United States of America", 100)
visualize_alliance_scores(us_preds, "United States")

/scratch/local/jobs/35877280/ipykernel_3700182/1584897772.py:18: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



## **Conclusion**

In this tutorial, we successfully built and trained a Graph Neural Network to predict missing and future links in the Cold War alliance network. We demonstrated how to transform raw edge data into a format suitable for edge prediction, build a GNN encoder-decoder model in PyTorch Geometric, and train it to achieve strong predictive performance.

Our key takeaways include:

1.  **GNNs Excel at Relational Learning:** By passing messages between nodes, GNNs can learn rich embeddings that capture a node's position and role within the wider network structure.
2.  **Edge Prediction is a Powerful Tool:** This technique moves beyond descriptive analytics into the realm of prediction, allowing us to forecast connections in social, biological, and technological networks.
3.  **PyTorch Geometric Simplifies GNNs:** The PyG library provides the high-level abstractions needed to build complex graph models efficiently.

This GNN-based approach provides a flexible and powerful foundation for a wide range of graph prediction tasks, offering a modern alternative to traditional methods like Node2Vec for creating predictive models of network dynamics.